In [6]:
%load_ext autoreload
%autoreload 2

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="4"
import torch
from mgtbench import AutoDetector, AutoExperiment
from mgtbench.loading.dataloader import load

/home/zhiyuan/miniconda3/envs/MGTBench/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = {'need_finetune': True,
          'need_save': False,
          'epochs': 1
        }

New dataset

In [3]:
model_name_or_path = '/data1/models/distilbert-base-uncased'
metric1 = AutoDetector.from_detector_name('LM-D', 
                                            model_name_or_path=model_name_or_path)
experiment = AutoExperiment.from_experiment_name('supervised',detector=[metric1])

2024-09-30 08:21:36.235286: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at /data1/models/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
category = ['Physics', 'Medicine', 'Biology', 'Electrical_engineering', 'Computer_science', 
            'Literature', 'History', 'Education', 'Art', 'Law', 'Management', 'Philosophy', 
            'Economy', 'Math', 'Statistics', 'Chemistry']
for cat in category:
    print(cat)
    datatype = cat
    llmname = 'Moonshot'
    data = load(datatype, llmname, cut_length=3000, disable=True, task='task3')
    data['train']['text'] = data['train']['text'][:1000]
    data['train']['label'] = data['train']['label'][:1000]
    data['test']['text'] = data['test']['text'][:]
    data['test']['label'] = data['test']['label'][:]
    print(len(data['train']['text']), len(data['test']['text']))

Physics
1000 1163
Medicine
1000 972
Biology
1000 1721
Electrical_engineering
1000 2181
Computer_science
1000 1670
Literature
1000 2223
History
1000 3271
Education
1000 1587
Art
1000 1017
Law
1000 854
Management
1000 394
Philosophy
1000 388
Economy
1000 960
Math
1000 1321
Statistics
1000 1028
Chemistry
1000 336


In [5]:
len(data['train']['text']), len(data['test']['text']), 

(1000, 336)

In [6]:
# manually cut the text length now
len(data['train']['text'][2])

1496

In [7]:
experiment.load_data(data)

In [8]:
res = experiment.launch(**config)

Calculate result for each data point
Running prediction of detector LM-D


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/home/zhiyuan/miniconda3/envs/MGTBench/lib/python3.8/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/home/zhiyuan/miniconda3/envs/MGTBench/lib/python3.8/site-packages/accelerate/accelerator.py:451: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
da

Step,Training Loss


Fine-tune finished
Predict training data


100%|██████████| 1000/1000 [00:05<00:00, 185.68it/s]


Predict testing data


100%|██████████| 336/336 [00:01<00:00, 188.30it/s]

Run classification for results


In [11]:
res[0].train, res[0].test

(Metric(acc=0.98, precision=0.9683168316831683, recall=0.9918864097363083, f1=0.9799599198396793, auc=0.9973214750091017),
 Metric(acc=0.9642857142857143, precision=0.9540229885057471, recall=0.9764705882352941, f1=0.9651162790697674, auc=0.9923104181431609))

In [12]:
torch.cuda.empty_cache()